# KishoLens ML & NLP Feature Extraction Prototype

This notebook prototypes the core NLP processing and feature extraction modules of KishoLens. It loads crawled chapter text from SQLite (`data/kisholens.db`), checks for advanced NLP libraries like `spacy`, `sudachipy`, `nltk`, and `hanlp`, and computes detailed stylistic pacing features (dependency tree depth, parts of speech distribution, pronoun/adjective ratios, vocabulary diversity, and pacing metrics) for English, Japanese, and Chinese texts. If advanced libraries are not available (e.g. on Python 3.14 due to dependency availability), the pipeline automatically falls back to regex-based baselines.

In [ ]:
import os
import re
import unicodedata
from typing import Optional, List, Dict, Any
from sqlmodel import SQLModel, Field, Session, create_engine, select
import pandas as pd

# Try importing NLP packages for high-fidelity extraction
try:
    import spacy
    HAS_SPACY = True
except ImportError:
    HAS_SPACY = False

try:
    import sudachipy
    HAS_SUDACHI = True
except ImportError:
    HAS_SUDACHI = False

try:
    import nltk
    HAS_NLTK = True
except ImportError:
    HAS_NLTK = False

try:
    import hanlp
    HAS_HANLP = False  # Disabled by default in prototype to prevent memory/JIT locks
except ImportError:
    HAS_HANLP = False

print(f"Libraries available: spaCy={HAS_SPACY}, SudachiPy={HAS_SUDACHI}, NLTK={HAS_NLTK}, HanLP={HAS_HANLP}")

In [ ]:
# Initialize NLP models and fallback flags dynamically

nlp_en = None
nlp_ja = None
nlp_zh = None
nlp_hanlp = None

def load_spacy_model(model_name: str):
    """Helper to dynamically load or download a spaCy model, removing loading redundancies."""
    try:
        return spacy.load(model_name)
    except Exception as e:
        print(f"Could not load spaCy model {model_name}: {e}")
        return None

if HAS_SPACY:
    import spacy
    if spacy.util.is_package("en_core_web_sm"):
        nlp_en = load_spacy_model("en_core_web_sm")
    if HAS_SUDACHI and spacy.util.is_package("ja_core_news_sm"):
        nlp_ja = load_spacy_model("ja_core_news_sm")
    if spacy.util.is_package("zh_core_web_sm"):
        nlp_zh = load_spacy_model("zh_core_web_sm")

if HAS_NLTK:
    try:
        import nltk
        nltk.download('punkt', quiet=True)
        nltk.download('punkt_tab', quiet=True)
        nltk.download('averaged_perceptron_tagger', quiet=True)
        nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    except Exception as e:
        print(f"Could not download NLTK resources: {e}")

if HAS_HANLP:
    try:
        import hanlp
        nlp_hanlp = hanlp.load(hanlp.pretrained.mtl.CLOSE_TOK_POS_NER_SRL_DEP_SDP_CON_ELECTRA_SMALL_ZH)
        print("Loaded HanLP Chinese Pipeline")
    except Exception as e:
        print(f"Could not initialize HanLP model: {e}")


In [ ]:
# Declare SQLModel tables

class Novel(SQLModel, table=True):
    __table_args__ = {"extend_existing": True}
    id: Optional[int] = Field(default=None, primary_key=True)
    title: str
    author: str
    source: str

class Chapter(SQLModel, table=True):
    __table_args__ = {"extend_existing": True}
    id: Optional[int] = Field(default=None, primary_key=True)
    novel_id: int = Field(foreign_key="novel.id")
    chapter_number: int
    title: str
    text_ja: str
    text_en: str
    text_zh: str = Field(default="")

In [ ]:
def compute_type_token_ratio(tokens: List[str]) -> float:
    """Computes Type-Token Ratio (vocabulary diversity)."""
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)


def compute_dep_tree_depth(doc) -> float:
    """
    Computes the average maximum dependency tree depth across all sentences in a spaCy document.
    """
    def get_depth(token):
        if not list(token.children):
            return 1
        return 1 + max(get_depth(child) for child in token.children)

    depths = []
    for sent in doc.sents:
        if sent.root:
            depths.append(get_depth(sent.root))
    return sum(depths) / len(depths) if depths else 0.0


def extract_english_features(text: str) -> Dict[str, Any]:
    """
    Extracts stylistic features from English text (with spaCy, NLTK, or Regex fallback).
    """
    if not text:
        return {}
        
    # Calculate baseline metrics on the FULL text using fast regex/string operations
    words = re.findall(r'\b\w+\b', text.lower())
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    dialogue_lines = [l for l in lines if l.startswith('"') or l.startswith("'") or l.startswith('“') or l.startswith('”')]
    punc_count = len(re.findall(r'[.,\/#!$%\^&\*;:{}=\-_`~()?"\']', text))
    
    word_count = len(words)
    sentence_count = len(sentences)
    avg_sentence_len = word_count / sentence_count if sentence_count > 0 else 0.0
    dialogue_ratio = len(dialogue_lines) / len(lines) if lines else 0.0
    punc_density = punc_count / len(text) if len(text) > 0 else 0.0
    
    # Fallback/default metrics
    dep_tree_depth = 0.0
    adj_ratio = 0.0
    verb_ratio = 0.0
    pron_ratio = 0.0
    entity_density = 0.0
    ttr = compute_type_token_ratio(words)
    
    # Extract advanced features on a sample limit of 10,000 characters
    if HAS_SPACY and nlp_en is not None:
        try:
            sample_text = text[:10000]
            doc = nlp_en(sample_text)
            sample_words = [t for t in doc if not t.is_punct and not t.is_space]
            sample_word_count = len(sample_words)
            if sample_word_count > 0:
                lemmas = [t.lemma_.lower() for t in sample_words]
                ttr = compute_type_token_ratio(lemmas)
                dep_tree_depth = compute_dep_tree_depth(doc)
                
                adj_count = len([t for t in doc if t.pos_ == "ADJ"])
                verb_count = len([t for t in doc if t.pos_ in ("VERB", "AUX")])
                pron_count = len([t for t in doc if t.pos_ == "PRON"])
                
                adj_ratio = adj_count / sample_word_count
                verb_ratio = verb_count / sample_word_count
                pron_ratio = pron_count / sample_word_count
                
                entity_count = len(doc.ents)
                entity_density = (entity_count / sample_word_count) * 100
        except Exception as e:
            print(f"Error in English spaCy features extraction: {e}")
            
    elif HAS_NLTK:
        try:
            import nltk
            sample_text = text[:10000]
            sample_words = nltk.word_tokenize(sample_text.lower())
            sample_word_count = len(sample_words)
            if sample_word_count > 0:
                tagged = nltk.pos_tag(sample_words)
                adj_count = len([w for w, tag in tagged if tag in ('JJ', 'JJR', 'JJS')])
                verb_count = len([w for w, tag in tagged if tag.startswith('V') or tag == 'MD'])
                pron_count = len([w for w, tag in tagged if tag in ('PRP', 'PRP$', 'WP', 'WP$')])
                
                adj_ratio = adj_count / sample_word_count
                verb_ratio = verb_count / sample_word_count
                pron_ratio = pron_count / sample_word_count
        except Exception:
            pass
            
    return {
        "word_count": word_count,
        "sentence_count": sentence_count,
        "avg_sentence_len": avg_sentence_len,
        "dialogue_ratio": dialogue_ratio,
        "ttr": ttr,
        "punc_density": punc_density,
        "dep_tree_depth": dep_tree_depth,
        "adj_ratio": adj_ratio,
        "verb_ratio": verb_ratio,
        "pron_ratio": pron_ratio,
        "entity_density": entity_density
    }

def extract_japanese_features(text: str) -> Dict[str, Any]:
    """
    Extracts stylistic features from Japanese text (with spaCy/Sudachi or Regex/NLTK fallback).
    """
    if not text:
        return {}
        
    # Calculate baseline metrics on the FULL text using fast string/regex operations
    chars = [c for c in text if not c.isspace()]
    sentences = [s.strip() for s in re.split(r'[。！？]+', text) if s.strip()]
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    dialogue_lines = [l for l in lines if l.startswith('「') or l.startswith('『')]
    punc_count = len(re.findall(r'[、。！？「」『』（）―…ー・]', text))
    
    char_count = len(chars)
    sentence_count = len(sentences)
    avg_sentence_len = char_count / sentence_count if sentence_count > 0 else 0.0
    dialogue_ratio = len(dialogue_lines) / len(lines) if lines else 0.0
    punc_density = punc_count / len(text) if len(text) > 0 else 0.0
    
    kanji_chars = re.findall(r'[\u4e00-\u9fff]', text)
    kanji_ratio = len(kanji_chars) / char_count if char_count > 0 else 0.0
    
    # Defaults
    dep_tree_depth = 0.0
    particle_ratio = 0.0
    verb_ratio = 0.0
    ttr = compute_type_token_ratio(chars)
    
    # Advanced features on a sample limit of 10,000 characters
    if HAS_SPACY and nlp_ja is not None:
        try:
            sample_text = text[:10000]
            doc = nlp_ja(sample_text)
            sample_words = [t for t in doc if not t.is_punct and not t.is_space]
            sample_word_count = len(sample_words)
            if sample_word_count > 0:
                lemmas = [t.lemma_ for t in sample_words]
                ttr = compute_type_token_ratio(lemmas)
                dep_tree_depth = compute_dep_tree_depth(doc)
                
                particle_count = len([t for t in doc if t.pos_ == "ADP" or "助词" in t.tag_ or t.tag_.startswith("助詞")])
                verb_count = len([t for t in doc if t.pos_ in ("VERB", "AUX") or "动词" in t.tag_ or t.tag_.startswith("動詞")])
                
                particle_ratio = particle_count / sample_word_count
                verb_ratio = verb_count / sample_word_count
        except Exception as e:
            print(f"Error in Japanese spaCy features extraction: {e}")
            
    return {
        "char_count": char_count,
        "sentence_count": sentence_count,
        "avg_sentence_len": avg_sentence_len,
        "dialogue_ratio": dialogue_ratio,
        "ttr": ttr,
        "punc_density": punc_density,
        "dep_tree_depth": dep_tree_depth,
        "particle_ratio": particle_ratio,
        "verb_ratio": verb_ratio,
        "kanji_ratio": kanji_ratio
    }

def extract_chinese_features(text: str) -> Dict[str, Any]:
    """
    Extracts stylistic features from Chinese text (utilizing spaCy, NLTK, HanLP, or Regex fallback).
    """
    if not text:
        return {}
        
    # Calculate baseline metrics on the FULL text using fast string/regex operations
    chars = [c for c in text if not c.isspace()]
    char_count = len(chars)
    sentences = [s.strip() for s in re.split(r'[。！？\n]+', text) if s.strip()]
    sentence_count = len(sentences)
    avg_sentence_len = char_count / sentence_count if sentence_count > 0 else 0.0
    
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    dialogue_lines = [l for l in lines if l.startswith('“') or l.startswith('「') or l.startswith('安全') or l.startswith('『')]
    dialogue_ratio = len(dialogue_lines) / len(lines) if lines else 0.0
    
    ttr = compute_type_token_ratio(chars)
    punc_count = len(re.findall(r'[，、。！？；：""‘’（）《》【】『』「」——……]', text))
    punc_density = punc_count / len(text) if len(text) > 0 else 0.0
    
    dep_tree_depth = 0.0
    particle_ratio = 0.0
    verb_ratio = 0.0
    
    # Advanced features on a sample limit of 10,000 characters
    if HAS_HANLP and nlp_hanlp is not None:
        try:
            sample_text = text[:10000]
            doc = nlp_hanlp(sample_text)
            words = doc.get('tok') or doc.get('tok/fine') or doc.get('tok/coarse')
            if words:
                word_count = len(words)
                ttr = compute_type_token_ratio(words)
                
                pos_tags = doc.get('pos') or doc.get('pos/pku') or doc.get('pos/ctb') or doc.get('pos/863')
                if pos_tags:
                    particle_count = sum(1 for tag in pos_tags if tag in ('u', 'y', '助词') or tag.startswith(('u', 'y')))
                    verb_count = sum(1 for tag in pos_tags if tag in ('v', 'vd', 'vn', '动词') or tag.startswith('v'))
                    particle_ratio = particle_count / word_count if word_count > 0 else 0.0
                    verb_ratio = verb_count / word_count if word_count > 0 else 0.0
                    
                heads = doc.get('dep')
                if heads:
                    def get_node_depth(idx, memo):
                        if idx in memo:
                            return memo[idx]
                        parent = heads[idx][0]
                        if parent == 0:
                            memo[idx] = 1
                            return 1
                        val = 1 + get_node_depth(parent - 1, memo)
                        memo[idx] = val
                        return val
                    memo = {}
                    depths = [get_node_depth(i, memo) for i in range(len(heads))]
                    dep_tree_depth = max(depths) if depths else 0.0
            return {
                "char_count": char_count,
                "sentence_count": sentence_count,
                "avg_sentence_len": avg_sentence_len,
                "dialogue_ratio": dialogue_ratio,
                "ttr": ttr,
                "punc_density": punc_density,
                "dep_tree_depth": dep_tree_depth,
                "particle_ratio": particle_ratio,
                "verb_ratio": verb_ratio
            }
        except Exception:
            pass
            
    if HAS_SPACY and nlp_zh is not None:
        try:
            sample_text = text[:10000]
            doc = nlp_zh(sample_text)
            words = [t for t in doc if not t.is_punct and not t.is_space]
            word_count = len(words)
            if word_count > 0:
                lemmas = [t.lemma_ for t in words]
                ttr = compute_type_token_ratio(lemmas)
                dep_tree_depth = compute_dep_tree_depth(doc)
                
                particle_count = len([t for t in doc if t.pos_ in ("ADP", "PART")])
                verb_count = len([t for t in doc if t.pos_ in ("VERB", "AUX")])
                
                particle_ratio = particle_count / word_count
                verb_ratio = verb_count / word_count
                
            return {
                "char_count": char_count,
                "sentence_count": sentence_count,
                "avg_sentence_len": avg_sentence_len,
                "dialogue_ratio": dialogue_ratio,
                "ttr": ttr,
                "punc_density": punc_density,
                "dep_tree_depth": dep_tree_depth,
                "particle_ratio": particle_ratio,
                "verb_ratio": verb_ratio
            }
        except Exception:
            pass
            
    return {
        "char_count": char_count,
        "sentence_count": sentence_count,
        "avg_sentence_len": avg_sentence_len,
        "dialogue_ratio": dialogue_ratio,
        "ttr": ttr,
        "punc_density": punc_density,
        "dep_tree_depth": dep_tree_depth,
        "particle_ratio": particle_ratio,
        "verb_ratio": verb_ratio
    }


In [ ]:
# Load Database and Process Chapters

db_path = "data/kisholens.db" if os.path.exists("data/kisholens.db") else "../data/kisholens.db"
engine = create_engine(f"sqlite:///{db_path}")

features_list = []

with Session(engine) as session:
    novels = session.exec(select(Novel)).all()
    novels_map = {n.id: n for n in novels}
    
    chapters = session.exec(select(Chapter)).all()
    print(f"Processing {len(chapters)} chapters...")
    
    for ch in chapters:
        novel = novels_map.get(ch.novel_id)
        if not novel:
            continue
            
        row = {
            "novel_title": novel.title,
            "author": novel.author,
            "source": novel.source,
            "chapter_num": ch.chapter_number,
            "chapter_title": ch.title
        }
        
        if ch.text_en:
            en_feat = extract_english_features(ch.text_en)
            row.update({f"en_{k}": v for k, v in en_feat.items()})
            
        if ch.text_ja:
            ja_feat = extract_japanese_features(ch.text_ja)
            row.update({f"ja_{k}": v for k, v in ja_feat.items()})

        if hasattr(ch, 'text_zh') and ch.text_zh:
            zh_feat = extract_chinese_features(ch.text_zh)
            row.update({f"zh_{k}": v for k, v in zh_feat.items()})
            
        features_list.append(row)

df = pd.DataFrame(features_list)
df.head()

In [ ]:
# Aggregate Stylistic Stats by Novel Source

print("Style Aggregates by Source Platform:")
group_cols = ["source"]
num_cols = [c for c in df.columns if c.startswith("en_") or c.startswith("ja_") or c.startswith("zh_")]
agg_df = df.groupby(group_cols)[num_cols].mean()
agg_df

In [ ]:
# Stylistic pacing comparisons (English texts)

en_cols = [c for c in df.columns if c.startswith("en_")]
if en_cols:
    print("English Style Aggregates by Novel:")
    novel_en = df.groupby(["novel_title", "source"])[en_cols].mean().dropna(how='all')
    print(novel_en.to_string())
else:
    print("No English style columns available.")

In [ ]:
# Stylistic pacing comparisons (Japanese texts)

ja_cols = [c for c in df.columns if c.startswith("ja_")]
if ja_cols:
    print("\nJapanese Style Aggregates by Novel:")
    novel_ja = df.groupby(["novel_title", "source"])[ja_cols].mean().dropna(how='all')
    print(novel_ja.to_string())
else:
    print("\nNo Japanese style columns available.")

In [ ]:
# Stylistic pacing comparisons (Chinese texts)

zh_cols = [c for c in df.columns if c.startswith("zh_")]
if zh_cols:
    print("\nChinese Style Aggregates by Novel:")
    novel_zh = df.groupby(["novel_title", "source"])[zh_cols].mean().dropna(how='all')
    print(novel_zh.to_string())
else:
    print("\nNo Chinese style columns available.")